In [1]:
# === INSTALLS (Colab) ===
!pip -q install datasets==2.21.0 transformers==4.44.2 torchmetrics==1.4.0 evaluate==0.4.3 accelerate==1.1.1

import sys, torch, transformers, datasets, torchmetrics, evaluate, accelerate
print(
    f"✅ versions -> torch {torch.__version__} | transformers {transformers.__version__} | "
    f"datasets {datasets.__version__} | torchmetrics {torchmetrics.__version__}"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 138.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.8/868.8 kB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
✅ versions -> torch 2.8.0+cu126 | transformers 4.44.2 | datasets 2.21.0 | torchmetrics 1.4.0


In [2]:
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from evaluate import load
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"
#  You can install and import any other libraries if needed

In [3]:
# Some Chinese punctuations will be tokenized as [UNK], so we replace them with English ones
token_replacement = [
    ["：" , ":"],
    ["，" , ","],
    ["“" , "\""],
    ["”" , "\""],
    ["？" , "?"],
    ["……" , "..."],
    ["！" , "!"]
]

In [4]:
tokenizer = BertTokenizer.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
class SemevalDataset(Dataset):
    def __init__(self, split="train") -> None:
        super().__init__()
        assert split in ["train", "validation", "test"]
        self.data = load_dataset(
            "sem_eval_2014_task_1", split=split, trust_remote_code=True, cache_dir="./cache/"
        ).to_list()

    def __getitem__(self, index):
        d = self.data[index]
        # Replace Chinese punctuations with English ones
        for k in ["premise", "hypothesis"]:
            for tok in token_replacement:
                d[k] = d[k].replace(tok[0], tok[1])
        return d

    def __len__(self):
        return len(self.data)

data_sample = SemevalDataset(split="train").data[:3]
print(f"Dataset example: \n{data_sample[0]} \n{data_sample[1]} \n{data_sample[2]}")

Generating train split:   0%|          | 0/4500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4927 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset example: 
{'sentence_pair_id': 1, 'premise': 'A group of kids is playing in a yard and an old man is standing in the background', 'hypothesis': 'A group of boys in a yard is playing and a man is standing in the background', 'relatedness_score': 4.5, 'entailment_judgment': 0} 
{'sentence_pair_id': 2, 'premise': 'A group of children is playing in the house and there is no man standing in the background', 'hypothesis': 'A group of kids is playing in a yard and an old man is standing in the background', 'relatedness_score': 3.200000047683716, 'entailment_judgment': 0} 
{'sentence_pair_id': 3, 'premise': 'The young boys are playing outdoors and the man is smiling nearby', 'hypothesis': 'The kids are playing outdoors near a man with a smile', 'relatedness_score': 4.699999809265137, 'entailment_judgment': 1}


In [6]:
# Define the hyperparameters
# You can modify these values if needed
lr = 2e-5
epochs = 30
train_batch_size = 32
validation_batch_size = 64

In [7]:
# TODO1: Create batched data for DataLoader
# `collate_fn` is a function that defines how the data batch should be packed.
# This function will be called in the DataLoader to pack the data batch.

def collate_fn(batch):
    # TODO1-1: Implement the collate_fn function
    # Write your code here
    # The input parameter is a data batch (tuple), and this function packs it into tensors.
    # Use tokenizer to pack tokenize and pack the data and its corresponding labels.
    # Return the data batch and labels for each sub-task.
    premise = [d["premise"] for d in batch]
    hypothesis = [d["hypothesis"] for d in batch]
    relatedness_score = torch.tensor([d["relatedness_score"] for d in batch], dtype=torch.float)
    entailment_judgment = torch.tensor([d["entailment_judgment"] for d in batch], dtype=torch.long)

    encoded_input = tokenizer(premise, hypothesis, padding=True, truncation=True, return_tensors="pt")
    return encoded_input, relatedness_score, entailment_judgment

# TODO1-2: Define your DataLoader
dl_train = DataLoader(SemevalDataset(split="train"), batch_size=train_batch_size, shuffle=True, collate_fn=collate_fn)
dl_validation = DataLoader(SemevalDataset(split="validation"), batch_size=validation_batch_size, shuffle=False, collate_fn=collate_fn)
dl_test = DataLoader(SemevalDataset(split="test"), batch_size=validation_batch_size, shuffle=False, collate_fn=collate_fn)

In [8]:
# TODO2: Construct your model
class MultiLabelModel(torch.nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Write your code here
        # Define what modules you will use in the model
        # Please use "google-bert/bert-base-uncased" model (https://huggingface.co/google-bert/bert-base-uncased)
        # Besides the base model, you may design additional architectures by incorporating linear layers, activation functions, or other neural components.
        # Remark: The use of any additional pretrained language models is not permitted.
        self.bert = BertModel.from_pretrained("google-bert/bert-base-uncased", cache_dir="./cache/")
        self.regressor = torch.nn.Linear(self.bert.config.hidden_size, 1)
        self.classifier = torch.nn.Linear(self.bert.config.hidden_size, 3) # 3 classes for entailment judgment (0, 1, 2)

    def forward(self, **kwargs):
        # Write your code here
        # Forward pass
        outputs = self.bert(**kwargs)
        pooled_output = outputs.pooler_output
        relatedness_score = self.regressor(pooled_output)
        entailment_judgment = self.classifier(pooled_output)
        return relatedness_score, entailment_judgment

In [9]:
# TODO3: Define your optimizer and loss function

model = MultiLabelModel().to(device)
# TODO3-1: Define your Optimizer
optimizer = AdamW(model.parameters(), lr=lr)

# TODO3-2: Define your loss functions (you should have two)
# Write your code here
criterion_reg = torch.nn.MSELoss() # Mean Squared Error for regression
criterion_cls = torch.nn.CrossEntropyLoss() # Cross Entropy for classification

# scoring functions
psr = load("pearsonr")
acc = load("accuracy")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [10]:
import os

best_score = 0.0
for ep in range(epochs):
    pbar = tqdm(dl_train)
    pbar.set_description(f"Training epoch [{ep+1}/{epochs}]")
    model.train()
    # TODO4: Write the training loop
    # Write your code here
    # train your model
    # clear gradient
    # forward pass
    # compute loss
    # back-propagation
    # model optimization
    running_loss = 0.0
    for batch in pbar:
        inputs, relatedness_score_labels, entailment_judgment_labels = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}
        relatedness_score_labels = relatedness_score_labels.to(device)
        entailment_judgment_labels = entailment_judgment_labels.to(device)

        optimizer.zero_grad()
        relatedness_score_pred, entailment_judgment_pred = model(**inputs)

        loss_reg = criterion_reg(relatedness_score_pred.squeeze(), relatedness_score_labels)
        loss_cls = criterion_cls(entailment_judgment_pred, entailment_judgment_labels)
        loss = loss_reg + loss_cls

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{running_loss/(pbar.n or 1):.4f}'})


    pbar = tqdm(dl_validation)
    pbar.set_description(f"Validation epoch [{ep+1}/{epochs}]")
    model.eval()
    # TODO5: Write the evaluation loop
    # Write your code here
    # Evaluate your model
    # Output all the evaluation scores (PearsonCorr, Accuracy)
    all_relatedness_scores_pred = []
    all_relatedness_scores_labels = []
    all_entailment_judgments_pred = []
    all_entailment_judgments_labels = []

    with torch.no_grad():
        for batch in pbar:
            inputs, relatedness_score_labels, entailment_judgment_labels = batch
            inputs = {k: v.to(device) for k, v in inputs.items()}
            relatedness_score_labels = relatedness_score_labels.to(device)
            entailment_judgment_labels = entailment_judgment_labels.to(device)

            relatedness_score_pred, entailment_judgment_pred = model(**inputs)

            all_relatedness_scores_pred.extend(relatedness_score_pred.squeeze().tolist())
            all_relatedness_scores_labels.extend(relatedness_score_labels.tolist())
            all_entailment_judgments_pred.extend(torch.argmax(entailment_judgment_pred, dim=1).tolist())
            all_entailment_judgments_labels.extend(entailment_judgment_labels.tolist())


    pearson_corr = psr.compute(predictions=all_relatedness_scores_pred, references=all_relatedness_scores_labels)["pearsonr"]
    accuracy = acc.compute(predictions=all_entailment_judgments_pred, references=all_entailment_judgments_labels)["accuracy"]
    print(f"Pearson Correlation: {pearson_corr:.4f}, Accuracy: {accuracy:.4f}")


    if pearson_corr + accuracy > best_score:
        best_score = pearson_corr + accuracy
        # Create the directory if it doesn't exist
        os.makedirs('./saved_models', exist_ok=True)
        torch.save(model.state_dict(), f'./saved_models/best_model.ckpt')

Validation epoch [1/20]: 100%|██████████| 8/8 [00:01<00:00,  6.38it/s]


Pearson Correlation: 0.8524, Accuracy: 0.8260


Validation epoch [2/20]: 100%|██████████| 8/8 [00:01<00:00,  5.94it/s]


Pearson Correlation: 0.8740, Accuracy: 0.8580


Validation epoch [3/20]: 100%|██████████| 8/8 [00:01<00:00,  5.42it/s]


...

Pearson Correlation: 0.8720, Accuracy: 0.8720


Validation epoch [19/20]: 100%|██████████| 8/8 [00:01<00:00,  5.67it/s]


Pearson Correlation: 0.8734, Accuracy: 0.8560


Validation epoch [20/20]: 100%|██████████| 8/8 [00:01<00:00,  5.42it/s]


Pearson Correlation: 0.8750, Accuracy: 0.8760


In [11]:
# Load the model
model = MultiLabelModel().to(device)
model.load_state_dict(torch.load(f"./saved_models/best_model.ckpt", weights_only=True))

# Test Loop
pbar = tqdm(dl_test, desc="Test")
model.eval()

# TODO6: Write the test loop
# Write your code here
# We have loaded the best model with the highest evaluation score for you
# Please implement the test loop to evaluate the model on the test dataset
# We will have 10% of the total score for the test accuracy and pearson correlation
import torch
import torch.nn as nn

def _enable_dropout(m):
    if isinstance(m, nn.Dropout):
        m.train()

N_MC = 16
CLAMP_MIN, CLAMP_MAX = 1.0, 5.0

model.apply(_enable_dropout)

all_relatedness_scores_pred = []
all_relatedness_scores_labels = []
all_entailment_judgments_pred = []
all_entailment_judgments_labels = []

with torch.no_grad():
    for batch in pbar:
        inputs, y_reg, y_cls = batch
        inputs = {k: v.to(device) for k, v in inputs.items()}
        y_reg = y_reg.to(device)
        y_cls = y_cls.to(device)


        reg_mc, cls_mc = [], []
        for _ in range(N_MC):
            reg_out, cls_logits = model(**inputs)
            reg_mc.append(reg_out.squeeze(-1))
            cls_mc.append(cls_logits)

        reg_pred = torch.stack(reg_mc, dim=0).mean(0)          # (B,)
        reg_pred = reg_pred.clamp(CLAMP_MIN, CLAMP_MAX)        # 範圍裁切 [1,5]
        cls_logits_mean = torch.stack(cls_mc, dim=0).mean(0)   # (B,3)
        cls_pred_ids = cls_logits_mean.argmax(dim=-1)          # (B,)


        all_relatedness_scores_pred.extend(reg_pred.cpu().tolist())
        all_relatedness_scores_labels.extend(y_reg.squeeze(-1).cpu().tolist())
        all_entailment_judgments_pred.extend(cls_pred_ids.cpu().tolist())
        all_entailment_judgments_labels.extend(y_cls.cpu().tolist())


pearson_corr = psr.compute(
    predictions=all_relatedness_scores_pred,
    references=all_relatedness_scores_labels
)["pearsonr"]
accuracy = acc.compute(
    predictions=all_entailment_judgments_pred,
    references=all_entailment_judgments_labels
)["accuracy"]

print(f"Test Pearson Correlation: {pearson_corr:.4f}, Test Accuracy: {accuracy:.4f}")

Test: 100%|██████████| 77/77 [03:04<00:00,  2.39s/it]


Test Pearson Correlation: 0.8901, Test Accuracy: 0.8794
